# Discrete Wavelet Transform Examples

Run examples to understand DWT.

## Import packages

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pywt

In [ ]:
# List all available wavelet families
print("Available wavelet families:")
print(pywt.families(short=False))
print(pywt.families(short=True))

# List all wavelets within the Daubechies family
print("\nWavelets in the Daubechies family:")
print(pywt.wavelist(family='bior'))

## Simple signal analysis using DWT

In [ ]:
# Generate the signal
t = np.linspace(0, 1, 1000, endpoint=False)
signal = np.cos(2.0 * np.pi * 7 * t) + np.sin(2.0 * np.pi * 13 * t)

# Apply DWT
coeffs = pywt.dwt(signal, 'db1') # db1: Daubechies family of wavelets order 1, aka. Haar wavelet
cA, cD = coeffs

# Plotting
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.plot(t, signal)
plt.title("Original Signal")
plt.subplot(1, 3, 2)
plt.plot(cA)
plt.title("Approximation Coefficients")
plt.subplot(1, 3, 3)
plt.plot(cD)
plt.title("Detail Coefficients")
plt.tight_layout()
plt.show()

## Visualizing Wavelet Transform of a Non-Stationary Signal

In [ ]:
# Generate a chirp signal
t = np.linspace(0, 1, 1000, endpoint=False)
signal = np.sin(2.0 * np.pi * 7 * t * t)

# Apply CWT
coefficients, frequencies = pywt.cwt(signal, scales=np.arange(1, 128), wavelet='cmor1.0-0.5')

# Plotting
# Create a figure with two subplots
fig, axs = plt.subplots(2, 1, figsize=(12, 8), gridspec_kw={'height_ratios': [1, 3]})

# Plot the original signal on the top subplot
axs[0].plot(t, signal)
axs[0].set_title("Original Chirp Signal")
axs[0].set_xlabel("Time")
axs[0].set_ylabel("Amplitude")
axs[0].grid(True, linestyle='--', alpha=0.6)

im = axs[1].imshow(np.abs(coefficients), 
                   aspect='auto', 
                   cmap='jet', 
                   origin='lower', # important to plot the data in the right order
                   extent=[0, 1, 1, 128])
axs[1].set_ylabel("Scale")
axs[1].set_xlabel("Time")
axs[1].set_title("CWT of a Chirp Signal")
# axs[1].invert_yaxis()

fig.colorbar(im, ax=axs[1], label="Magnitude")
plt.show()

## Denoising a signal

In [ ]:
# Generate a simple sinusoidal signal
t = np.linspace(0, 1, 1000, endpoint=False)
clean_signal = np.sin(2.0 * np.pi * 10 * t)

# Add random noise
noise = np.random.normal(0, 0.5, clean_signal.shape)
noisy_signal = clean_signal + noise

# Plotting the clean and noisy signals
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(t, clean_signal)
plt.title("Clean Signal")
plt.subplot(1, 2, 2)
plt.plot(t, noisy_signal)
plt.title("Noisy Signal")
plt.tight_layout()
plt.show()

In [ ]:
# Perform a multi-level wavelet decomposition
coeffs = pywt.wavedec(noisy_signal, 'db4', level=4)

# Set a threshold to nullify smaller coefficients (assumed to be noise)
threshold = 0.5
coeffs_thresholded = [pywt.threshold(c, threshold, mode='soft') for c in coeffs]

# Reconstruct the signal from the thresholded coefficients
denoised_signal = pywt.waverec(coeffs_thresholded, 'db4')

# Plotting the noisy and denoised signals
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(t, noisy_signal)
plt.title("Noisy Signal")
plt.subplot(1, 2, 2)
plt.plot(t, denoised_signal)
plt.title("Denoised Signal")
plt.tight_layout()
plt.show()

In [ ]:
# Perform a multi-level wavelet decomposition with both wavelets
coeffs_db1 = pywt.wavedec(noisy_signal, 'db1', level=4)
coeffs_db3 = pywt.wavedec(noisy_signal, 'db3', level=4)

# Set a threshold to nullify smaller coefficients
threshold = 0.5
coeffs_db1_thresh = [pywt.threshold(c, threshold, mode='soft') for c in coeffs_db1]
coeffs_db3_thresh = [pywt.threshold(c, threshold, mode='soft') for c in coeffs_db3]

# Reconstruct the signals
denoised_signal_db1 = pywt.waverec(coeffs_db1_thresh, 'db1')
denoised_signal_db3 = pywt.waverec(coeffs_db3_thresh, 'db3')

# Plotting the results for comparison
plt.figure(figsize=(15, 10))

# Original Clean Signal
plt.subplot(3, 1, 1)
plt.plot(t, clean_signal, 'g', label='Original Clean Signal')
plt.title("Original Signal")
plt.legend()

# Noisy Signal
plt.subplot(3, 1, 2)
plt.plot(t, noisy_signal, 'c', label='Noisy Signal', alpha=0.6)
plt.title("Noisy Signal")
plt.legend()

# Denoised Signals
plt.subplot(3, 1, 3)
plt.plot(t, denoised_signal_db1, 'r', label='Denoised with db1 (Haar)', linewidth=2)
plt.plot(t, denoised_signal_db3, 'b', label='Denoised with db3', linewidth=2)
plt.title("Comparison of Denoised Signals")
plt.legend()

plt.tight_layout()
plt.show()

## Image Compression using Wavelet Transform

In [ ]:
# Generate a simple 2D image
x, y = np.mgrid[0:1:128j, 0:1:128j]
img = np.sin(2.0 * np.pi * 7 * x) + np.sin(2.0 * np.pi * 13 * y)

# Plotting the generated image
plt.figure(figsize=(5, 5))
plt.imshow(img, cmap='gray')
plt.title("Generated Image")
plt.colorbar()
plt.show()

In [ ]:
coeffs2 = pywt.wavedec2(img, 'db3', level=3)

# Flatten and concatenate all the 2D coefficients
all_coeffs = np.concatenate([c.ravel() for sublist in coeffs2 for c in sublist if isinstance(c, np.ndarray)])

# Determine a threshold to retain a certain percentage of the strongest coefficients
percent = 10  # retain only the top 10% of coefficients
threshold = np.percentile(np.abs(all_coeffs), 100 - percent)

# Threshold the coefficients
coeffs2_thresholded = [(pywt.threshold(c, threshold, mode='soft') if isinstance(c, np.ndarray) else c) 
                       for c in coeffs2]

# Reconstruct the compressed image
compressed_img = pywt.waverec2(coeffs2_thresholded, 'db3')

# Plotting the original and compressed images
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(img, cmap='gray')
plt.title("Original Image")
plt.colorbar()
plt.subplot(1, 2, 2)
plt.imshow(compressed_img, cmap='gray')
plt.title("Compressed Image")
plt.colorbar()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pywt

# Generate a simple 2D image
x, y = np.mgrid[0:1:128j, 0:1:128j]
img = np.sin(2.0 * np.pi * 7 * x) + np.sin(2.0 * np.pi * 13 * y)

# --- Compression Parameters ---
percent_to_keep = 10  # Retain only the top 10% of coefficients
wavelet = 'db4'       # A slightly smoother wavelet than 'db1'

# --- Process for Level 1 ---
coeffs_l1 = pywt.wavedec2(img, wavelet, level=1)
# Correctly flatten coefficients for wavedec2 structure
all_coeffs_l1 = np.concatenate([coeffs_l1[0].ravel()] + [d.ravel() for d in coeffs_l1[1]])
threshold_l1 = np.percentile(np.abs(all_coeffs_l1), 100 - percent_to_keep)
coeffs_l1_thresh = [pywt.threshold(c, threshold_l1, mode='soft') if isinstance(c, np.ndarray) else c for c in [coeffs_l1[0]] + list(coeffs_l1[1])]
# Rebuild the coefficient list structure for waverec2
coeffs_l1_thresh_structured = [coeffs_l1_thresh[0], tuple(coeffs_l1_thresh[1:])]
compressed_img_l1 = pywt.waverec2(coeffs_l1_thresh_structured, wavelet)


# --- Process for Level 4 ---
coeffs_l4 = pywt.wavedec2(img, wavelet, level=4)
# Correctly flatten coefficients for wavedec2 structure
all_coeffs_l4 = np.concatenate([coeffs_l4[0].ravel()] + [d.ravel() for level_coeffs in coeffs_l4[1:] for d in level_coeffs])
threshold_l4 = np.percentile(np.abs(all_coeffs_l4), 100 - percent_to_keep)
# Threshold all coefficient arrays
coeffs_l4_thresh_flat = [pywt.threshold(c, threshold_l4, mode='soft') if isinstance(c, np.ndarray) else c for sublist in coeffs_l4 for c in (sublist if isinstance(sublist, tuple) else [sublist])]
# Rebuild the nested structure for waverec2
coeffs_l4_thresh_structured = [coeffs_l4_thresh_flat[0]] + [tuple(coeffs_l4_thresh_flat[i:i+3]) for i in range(1, len(coeffs_l4_thresh_flat), 3)]
compressed_img_l4 = pywt.waverec2(coeffs_l4_thresh_structured, wavelet)


# --- Plotting the results ---
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.imshow(img, cmap='gray')
plt.title("Original Image")

plt.subplot(1, 3, 2)
plt.imshow(compressed_img_l1, cmap='gray')
plt.title(f"Compressed (level=1, {percent_to_keep}% coeffs)")

plt.subplot(1, 3, 3)
plt.imshow(compressed_img_l4, cmap='gray')
plt.title(f"Compressed (level=4, {percent_to_keep}% coeffs)")

plt.tight_layout()
plt.show()


## Edge detection using DWT

In [ ]:
from skimage import data

# Use a built-in image from scikit-image as an example
img_photo = data.camera()

# Plotting the original image
plt.figure(figsize=(5, 5))
plt.imshow(img_photo, cmap='gray')
plt.title("Original Image")
plt.colorbar()
plt.show()

In [ ]:
# Perform a 2D wavelet decomposition on the image
coeffs_photo = pywt.wavedec2(img_photo, 'db1', level=1)
cA_photo, (cH_photo, cV_photo, cD_photo) = coeffs_photo

# Plotting the detail coefficients (edges)
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.imshow(np.abs(cH_photo), cmap='gray')
plt.title("Horizontal Detail (Edges)")
plt.colorbar()
plt.subplot(1, 3, 2)
plt.imshow(np.abs(cV_photo), cmap='gray')
plt.title("Vertical Detail (Edges)")
plt.colorbar()
plt.subplot(1, 3, 3)
plt.imshow(np.abs(cD_photo), cmap='gray')
plt.title("Diagonal Detail (Edges)")
plt.colorbar()
plt.tight_layout()
plt.show()

In [ ]:
from skimage import data
import numpy as np
import matplotlib.pyplot as plt
import pywt

# Use a built-in image from scikit-image as an example
img_photo = data.camera()

# --- Parameters to Compare ---
wavelet_db1 = 'db1'
wavelet_db8 = 'db8'
level = 4

# --- Perform Decomposition ---
coeffs_db1 = pywt.wavedec2(img_photo, wavelet_db1, level=level)
coeffs_db8 = pywt.wavedec2(img_photo, wavelet_db8, level=level)

# --- Plotting the Results ---
fig, axes = plt.subplots(2, level, figsize=(16, 8))
fig.suptitle("Comparison of Wavelet Details at Different Levels", fontsize=16)

for i in range(level):
    # Access details from finest (level 1) to coarsest (level 4)
    # The list is ordered [cA4, (cH4,cV4,cD4), (cH3,cV3,cD3), (cH2,cV2,cD2), (cH1,cV1,cD1)]
    # So, index -1 is level 1, -2 is level 2, etc.
    level_index = i + 1
    cH_db1, cV_db1, cD_db1 = coeffs_db1[-level_index]
    cH_db8, cV_db8, cD_db8 = coeffs_db8[-level_index]

    print(level_index)
    # Combine the absolute values of the three detail coefficients for a single edge map
    edge_map_db1 = np.abs(cH_db1) + np.abs(cV_db1) + np.abs(cD_db1)
    edge_map_db8 = np.abs(cH_db8) + np.abs(cV_db8) + np.abs(cD_db8)
    print(edge_map_db1.shape)
    
    ax = axes[0, i]
    ax.imshow(edge_map_db1, cmap='gray')
    ax.set_title(f"'{wavelet_db1}' - Level {level_index} Details")
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Plot the edge map for the current level
    ax = axes[1, i]
    ax.imshow(edge_map_db8, cmap='gray')
    ax.set_title(f"'{wavelet_db8}' - Level {level_index} Details")
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()


In [ ]:
from image_processing_tools.util.load_files import find_files_by_pattern
from image_processing_tools.image_class.image_container import ImageContainer

search_path = (
    "~/A8/Data_Roan/250704_CA_Ca9_WT_smFISH_Cocu_SSU_GAPDH/CA_SSU_GAPDH_Infection_FOV1_1_CY5, CY3.5 NAR, FITC, DAPI, BF",
    "~/A8/Data_Roan/250704_CA_Ca9_WT_smFISH_Cocu_SSU_GAPDH/CA_SSU_GAPDH_Infection_FOV2_1_CY5, CY3.5 NAR, FITC, DAPI",
    "~/A8/Data_Roan/250925_Cocu_Dyes_cFOS_CSF3_ECE1/Cocu_Cellbrite_cFOS_CSF3_ECE1_01_CY5, CY3.5 NAR, CY3, FITC, DAPI",
    "~/A8/Data_Roan/251114_MonoCa9_Phalloidin/CA9_Phalloidin_01_CY5, FITC, DAPI",
    "~/A8/Data_Roan/251114_MonoCa9_Phalloidin/CA9_Phalloidin_01_CY5, FITC, DAPI, BF"
    )

file_pattern = (
    "*MAX*.tif",
    "*SHARPEST-normalized_variance*.tif",
    "*SHARPEST-manual*.tif",
    "SUBTRACT-direct*O5*.tif"
    )

config = {
    "preprocessing": {
        "resize_image": False,
        "max_dim": 1024,
        "outlier_percentile": 0.35,
        "quantization": "8bit"
    }
}

# Find the files. The 'files' variable will hold the list of Path objects.
files = find_files_by_pattern(search_path[3], file_pattern[2], verbose=True)

cy5_container = ImageContainer(files[0],config)
cy5_image = cy5_container.merge()
cy5_dapi = ImageContainer([files[0],files[2]],config)

In [ ]:
from skimage import data
import numpy as np
import matplotlib.pyplot as plt
import pywt

# Use a built-in image from scikit-image as an example
img_photo = cy5_image

# --- Parameters to Compare ---
wavelet_db1 = 'db1'
wavelet_db8 = 'db8'
level = 4

# --- Perform Decomposition ---
coeffs_db1 = pywt.wavedec2(img_photo, wavelet_db1, level=level)
coeffs_db8 = pywt.wavedec2(img_photo, wavelet_db8, level=level)

# --- Plotting the Results ---
fig, axes = plt.subplots(2, level, figsize=(16, 8))
fig.suptitle("Comparison of Wavelet Details at Different Levels", fontsize=16)

for i in range(level):
    # Access details from finest (level 1) to coarsest (level 4)
    # The list is ordered [cA4, (cH4,cV4,cD4), (cH3,cV3,cD3), (cH2,cV2,cD2), (cH1,cV1,cD1)]
    # So, index -1 is level 1, -2 is level 2, etc.
    level_index = i + 1
    cH_db1, cV_db1, cD_db1 = coeffs_db1[-level_index]
    cH_db8, cV_db8, cD_db8 = coeffs_db8[-level_index]

    print(level_index)
    # Combine the absolute values of the three detail coefficients for a single edge map
    edge_map_db1 = np.abs(cH_db1) + np.abs(cV_db1) + np.abs(cD_db1)
    edge_map_db8 = np.abs(cH_db8) + np.abs(cV_db8) + np.abs(cD_db8)
    print(edge_map_db1.shape)
    
    ax = axes[0, i]
    ax.imshow(edge_map_db1, cmap='gray')
    ax.set_title(f"'{wavelet_db1}' - Level {level_index} Details")
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Plot the edge map for the current level
    ax = axes[1, i]
    ax.imshow(edge_map_db8, cmap='gray')
    ax.set_title(f"'{wavelet_db8}' - Level {level_index} Details")
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()


In [ ]:
from skimage import data
import numpy as np
import matplotlib.pyplot as plt
import pywt

# Use a built-in image from scikit-image as an example
img_photo = cy5_image

# --- Parameters to Compare ---
wavelet_db1 = 'db1'
wavelet_db8 = 'db8'
level = 4
cutoff_percentage = 50  # Percentile of coefficients to zero out (e.g., 95 removes bottom 95%)

# --- Perform Decomposition ---
coeffs_db1 = pywt.wavedec2(img_photo, wavelet_db1, level=level)
coeffs_db8 = pywt.wavedec2(img_photo, wavelet_db8, level=level)

# --- Plotting the Results ---
fig, axes = plt.subplots(2, level, figsize=(16, 8))
fig.suptitle(f"Comparison of Denoised Wavelet Details (Cutoff: {cutoff_percentage}%)", fontsize=16)

for i in range(level):
    # Access details from finest (level 1) to coarsest (level 4)
    # The list is ordered [cA4, (cH4,cV4,cD4), (cH3,cV3,cD3), (cH2,cV2,cD2), (cH1,cV1,cD1)]
    # So, index -1 is level 1, -2 is level 2, etc.
    level_index = i + 1
    cH_db1, cV_db1, cD_db1 = coeffs_db1[-level_index]
    cH_db8, cV_db8, cD_db8 = coeffs_db8[-level_index]

    # --- Denoising: Calculate Threshold based on Percentile ---
    # We combine the coefficients to find the global percentile for this level
    details_db1 = np.concatenate([np.abs(cH_db1).ravel(), np.abs(cV_db1).ravel(), np.abs(cD_db1).ravel()])
    thresh_db1 = np.percentile(details_db1, cutoff_percentage)
    
    details_db8 = np.concatenate([np.abs(cH_db8).ravel(), np.abs(cV_db8).ravel(), np.abs(cD_db8).ravel()])
    thresh_db8 = np.percentile(details_db8, cutoff_percentage)

    # --- Apply Thresholding ---
    cH_db1 = pywt.threshold(cH_db1, thresh_db1, mode='soft')
    cV_db1 = pywt.threshold(cV_db1, thresh_db1, mode='soft')
    cD_db1 = pywt.threshold(cD_db1, thresh_db1, mode='soft')

    cH_db8 = pywt.threshold(cH_db8, thresh_db8, mode='soft')
    cV_db8 = pywt.threshold(cV_db8, thresh_db8, mode='soft')
    cD_db8 = pywt.threshold(cD_db8, thresh_db8, mode='soft')

    print(f"Level {level_index} - Threshold db1: {thresh_db1:.4f}, db8: {thresh_db8:.4f}")

    # Combine the absolute values of the three detail coefficients for a single edge map
    edge_map_db1 = np.abs(cH_db1) + np.abs(cV_db1) + np.abs(cD_db1)
    edge_map_db8 = np.abs(cH_db8) + np.abs(cV_db8) + np.abs(cD_db8)
    print(edge_map_db1.shape)
    
    ax = axes[0, i]
    ax.imshow(edge_map_db1, cmap='gray')
    ax.set_title(f"'{wavelet_db1}' - Level {level_index} Details")
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Plot the edge map for the current level
    ax = axes[1, i]
    ax.imshow(edge_map_db8, cmap='gray')
    ax.set_title(f"'{wavelet_db8}' - Level {level_index} Details")
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()